In [ ]:
%pip install webdriver-manager requests beautifulsoup4 lxml selenium pandas

In [ ]:
url = 'https://auto.danawa.com/auto/?Work=record'


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get(url)

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div > div:nth-child(3) > div.left > table > tbody')
result = result.select('tr')
for tag in result:
    print(tag.select_one('a').text.strip(), tag.select_one('.num').text)
    #print(tag.select_one('.num').text)

#종료
driver.quit()

기아 42,066
현대 40,066
제네시스 6,942
KGM 3,701
르노코리아 2,000


In [ ]:
# 다나와 2025년 1월부터 12월까지 국산 자동차 판매 대수 수집


In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

company, amount = [], []
year, month = [], []
for _month in range(1,3):
    _year = '2026'
    _url = f'https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month={_year}-{_month:02}-00'

    # 페이지 열기
    driver.get(_url)
    time.sleep(1)  # 페이지 완전 로딩될때까지 기다림.. 대략 1초면 충분하다고 판단

    html = driver.page_source   # 정적에서 response.text 와 동일
    soup = BeautifulSoup(html, 'lxml')
    result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div:nth-child(1) > table > tbody')
    result = result.select('tr')
    count = len(result)
    for tag in result:        
        # print(tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] )
        co, am =  tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] 
        company.append(co); amount.append(am)
    year += [_year]*count
    month += [_month]*count
#종료
driver.quit()

In [14]:
import pandas as pd
data = {
    'year' : year,
    'month' : month,
    'company' : company,
    'amount' : amount
}
df = pd.DataFrame(data)
df['amount'] = df['amount'].str.replace(',','').astype(int)
df.head()

,year,month,company,amount
0,2026,1,기아,43129
1,2026,1,현대,41537
2,2026,1,제네시스,8671
3,2026,1,KGM,3187
4,2026,1,르노코리아,2239


In [15]:
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import os
# 환경변수 읽어오기
load_dotenv()

# 환경변수에서 MySQL 연결 정보 로드
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'car_db'),
    'port': int(os.getenv('DB_PORT', 3306))
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
query = '''
insert into sale_car(year,month,company,amount) 
values(%s,%s,%s,%s)
'''

for _, row in df.iterrows():
    params = (row.year, row.month,row.company, row.amount)
    cursor.execute(query,params=params)

conn.commit()

cursor.close()
conn.close()
print('데이터 삽입 완료')

데이터 삽입 완료


In [16]:
for _, row in df.iterrows():
    print(row.year, row.month, row.company, row.amount)

2026 1 기아 43129
2026 1 현대 41537
2026 1 제네시스 8671
2026 1 KGM 3187
2026 1 르노코리아 2239
2026 1 쉐보레 732
2026 2 기아 42066
2026 2 현대 40066
2026 2 제네시스 6942
2026 2 KGM 3701
2026 2 르노코리아 2000
2026 2 쉐보레 909


In [3]:
import os
import time
from dotenv import load_dotenv
import mysql.connector

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# 1. DB 설정
# ---------------------------------------------------------
# 환경변수 강제 업데이트
load_dotenv(override=True)

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'faq_data'), 
    'port': int(os.getenv('DB_PORT', 3306))
}

print("DB 설정 확인:", DB_CONFIG)
print("데이터베이스에 연결 중...")
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

insert_query = '''
INSERT INTO hyundai_faq (category_main, category_sub, question, answer) 
VALUES (%s, %s, %s, %s)
'''

# ---------------------------------------------------------
# 2. 브라우저 실행 및 접속
# ---------------------------------------------------------
print("크롬 브라우저를 실행합니다...")
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized') 
options.add_argument('--window-size=1920,1080')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

url = 'https://www.hyundai.com/kr/ko/faq.html'
driver.get(url)
time.sleep(2)

try:
    # 대분류 리스트 추출
    main_select_elem = wait.until(EC.presence_of_element_located((By.ID, 'category_depth1')))
    main_select = Select(main_select_elem)
    main_options = [opt.text for opt in main_select.options if opt.get_attribute('value')]

    # ---------------------------------------------------------
    # 3. 크롤링 메인 루프 
    # ---------------------------------------------------------
    for main_text in main_options:
        try: 
            # 대분류 선택
            Select(driver.find_element(By.ID, 'category_depth1')).select_by_visible_text(main_text)
            time.sleep(1) 
            
            # 바로 확인(조회) 버튼 클릭
            confirm_btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, '#contents > div.faq > div > div.section_white > div > form > fieldset > button'))
            )
            confirm_btn.click()
            
            page_num = 1
            while True:
                # 리스트가 화면에 렌더링될 때까지 대기
                try:
                    WebDriverWait(driver, 5).until(
                        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.ui_accordion.acc_01 dl'))
                    )
                    time.sleep(1.5) # 이미지 로딩을 위한 여유 시간
                except:
                    print(f"[{main_text}] {page_num}페이지 데이터를 불러오지 못했습니다. 다음으로 넘어갑니다.")
                    break
                
                # HTML 파싱
                soup = BeautifulSoup(driver.page_source, 'lxml')
                faq_list = soup.select('div.ui_accordion.acc_01 dl')
                
                page_data_batch = []

                for item in faq_list:
                    title_tag = item.select_one('.title')
                    content_tag = item.select_one('dd')
                    
                    if title_tag and content_tag:
                        q_raw = title_tag.text.strip()
                        
                        # ✨ 핵심 추가: 텍스트를 뽑기 전에, img 태그를 찾아서 [이미지: 주소] 형태로 치환 ✨
                        images = content_tag.find_all('img')
                        for img in images:
                            img_src = img.get('src') or img.get('data-src')
                            if img_src:
                                # 현대자동차 사이트 특성상 앞부분 도메인이 빠져있을 수 있으므로 채워줌
                                if img_src.startswith('/'):
                                    img_src = 'https://www.hyundai.com' + img_src
                                elif not img_src.startswith('http'):
                                    img_src = 'https://www.hyundai.com/' + img_src
                                
                                cleaned_url = img_src.strip()
                                # Streamlit 마크다운 오작동 방지
                                if cleaned_url.endswith('-') or cleaned_url.endswith('='):
                                    cleaned_url += ' '
                                
                                # img 태그 자체를 텍스트로 덮어씌움
                                img.replace_with(f"\n\n[이미지: {cleaned_url}]\n\n")

                        # 치환이 끝난 후 전체 텍스트 추출 (줄바꿈 유지)
                        a_raw = content_tag.get_text(separator='\n', strip=True) 
                        
                        cat_main, cat_sub = main_text, "분류없음"
                        real_q = q_raw
                        
                        # 카테고리 분리 로직 (현대차 특성)
                        if '[' in q_raw and ']' in q_raw:
                            start_idx, end_idx = q_raw.find('['), q_raw.find(']')
                            category_part = q_raw[start_idx+1 : end_idx]
                            real_q = q_raw[end_idx+1 : ].strip()
                            
                            if '>' in category_part:
                                extracted_main, extracted_sub = category_part.split('>', 1)
                                cat_main, cat_sub = extracted_main.strip(), extracted_sub.strip()
                            else:
                                cat_main = category_part.strip()
                        
                        page_data_batch.append((cat_main, cat_sub, real_q, a_raw))
                
                # DB에 Insert
                if page_data_batch:
                    cursor.executemany(insert_query, page_data_batch)
                    conn.commit()

                print(f"✅ [{main_text}] {page_num}페이지 수집 완료 (추출된 데이터: {len(page_data_batch)}개 / 이미지 처리됨)")

                # 다음 페이지 이동
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, 'button.navi.next')
                    if next_btn.is_enabled() and next_btn.is_displayed():
                        driver.execute_script("arguments[0].click();", next_btn)
                        page_num += 1
                        time.sleep(1.5) 
                    else:
                        break 
                except:
                    break 

        except Exception as e:
            print(f"🚨 [{main_text}] 수집 중 에러 발생: {e}")
            driver.get(url) 
            time.sleep(2)
            continue

finally:
    print("\n🎉 모든 크롤링 작업 완료. 자원을 정리합니다.")
    driver.quit()
    if cursor: cursor.close()
    if conn.is_connected(): conn.close()
    print("현대자동차 데이터 삽입 및 정리 완료")

DB 설정 확인: {'host': 'localhost', 'user': 'root', 'password': 'qwer123456@@', 'database': 'faq_data', 'port': 3306}
데이터베이스에 연결 중...
크롬 브라우저를 실행합니다...
✅ [차량구매] 1페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량구매] 2페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량구매] 3페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량구매] 4페이지 수집 완료 (추출된 데이터: 2개 / 이미지 처리됨)
✅ [차량정비] 1페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량정비] 2페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량정비] 3페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량정비] 4페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [차량정비] 5페이지 수집 완료 (추출된 데이터: 6개 / 이미지 처리됨)
✅ [정비예약] 1페이지 수집 완료 (추출된 데이터: 1개 / 이미지 처리됨)
✅ [홈페이지] 1페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [홈페이지] 2페이지 수집 완료 (추출된 데이터: 1개 / 이미지 처리됨)
✅ [모젠서비스] 1페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [모젠서비스] 2페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [모젠서비스] 3페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [모젠서비스] 4페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [모젠서비스] 5페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [모젠서비스] 6페이지 수집 완료 (추출된 데이터: 10개 / 이미지 처리됨)
✅ [모젠서비스] 7페이지 수집 완료 (추출된 데이터: 10개 / 이미지

In [2]:
import os
import time
from dotenv import load_dotenv
import mysql.connector

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# 1. DB 설정 
# ---------------------------------------------------------
load_dotenv(override=True)
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'faq_data'), 
    'port': int(os.getenv('DB_PORT', 3306))
}

print("데이터베이스에 연결 중... (DB:", DB_CONFIG['database'], ")")
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

insert_query = '''
INSERT INTO kia_faq (category, question, answer) 
VALUES (%s, %s, %s)
'''

# ---------------------------------------------------------
# 2. 브라우저 실행 및 접속
# ---------------------------------------------------------
print("크롬 브라우저를 실행합니다...")
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized') 
options.add_argument('--window-size=1920,1080')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

url = 'https://www.kia.com/kr/customer-service/center/faq'
driver.get(url)
time.sleep(3) 

target_categories = ['차량 구매', '차량 정비', '기아멤버스', '홈페이지', 'PBV', '기타']

try:
    # ---------------------------------------------------------
    # 3. 크롤링 메인 루프
    # ---------------------------------------------------------
    for category in target_categories:
        try:
            print(f"\n[{category}] 카테고리 수집 준비 중...")
            
            tab_xpath = f"//*[contains(text(), '{category}') and (self::button or self::a or self::li or self::span)]"
            tab_elems = wait.until(EC.presence_of_all_elements_located((By.XPATH, tab_xpath)))
            
            clicked = False
            for elem in tab_elems:
                if elem.is_displayed(): 
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elem)
                    time.sleep(0.5)
                    driver.execute_script("arguments[0].click();", elem)
                    clicked = True
                    break
            
            if not clicked:
                print(f"🚨 [{category}] 화면에서 해당 탭을 찾지 못했습니다. 건너뜁니다.")
                continue
                
            time.sleep(2) 
            
            page_num = 1
            while True:
                # 데이터 로딩 대기
                try:
                    WebDriverWait(driver, 5).until(
                        EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.cmp-accordion__title'))
                    )
                    time.sleep(1.5)
                except:
                    print(f"[{category}] {page_num}페이지 데이터를 찾을 수 없습니다. (마지막 페이지)")
                    break
                
                # HTML 파싱 
                soup = BeautifulSoup(driver.page_source, 'lxml')
                titles = soup.select('.cmp-accordion__title')
                panels = soup.select('.cmp-accordion__panel')
                
                page_data_batch = []
                for t_tag, p_tag in zip(titles, panels):
                    q_text = t_tag.text.strip()
                    
                    # ✨ 핵심 추가: 텍스트를 뽑기 전에, img 태그를 찾아서 [이미지: 주소] 형태로 치환
                    images = p_tag.find_all('img')
                    for img in images:
                        img_src = img.get('src')
                        if img_src:
                            # 기아자동차 사이트 특성상 앞부분 도메인이 빠져있을 수 있으므로 채워줌
                            if img_src.startswith('/'):
                                img_src = 'https://www.kia.com' + img_src
                            elif not img_src.startswith('http'):
                                img_src = 'https://www.kia.com/' + img_src
                            
                            # img 태그 자체를 텍스트로 덮어씌움
                            img.replace_with(f"\n\n[이미지: {img_src}]\n\n")

                    # 치환이 끝난 후 전체 텍스트 추출
                    a_text = p_tag.get_text(separator='\n', strip=True)
                    page_data_batch.append((category, q_text, a_text))
                
                # DB 저장
                if page_data_batch:
                    cursor.executemany(insert_query, page_data_batch)
                    conn.commit()
                
                print(f"✅ [{category}] {page_num}페이지 수집 완료 (추출된 데이터: {len(page_data_batch)}개)")
                
                # ---------------------------------------------------------
                # 4. 스마트 페이징 처리 
                # ---------------------------------------------------------
                target_num = str(page_num + 1)
                
                num_xpath = f"//*[(self::a or self::button) and normalize-space()='{target_num}']"
                num_btns = driver.find_elements(By.XPATH, num_xpath)
                
                if num_btns and num_btns[0].is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", num_btns[0])
                    time.sleep(0.5)
                    driver.execute_script("arguments[0].click();", num_btns[0])
                    page_num += 1
                    time.sleep(1.5)
                    continue
                else:
                    next_arrow_xpath = "//*[(self::a or self::button) and (contains(@class, 'next') or contains(@class, 'Next') or contains(@aria-label, '다음') or contains(@title, '다음'))]"
                    arrows = driver.find_elements(By.XPATH, next_arrow_xpath)
                    
                    clicked_arrow = False
                    for arrow in arrows:
                        if arrow.is_displayed() and arrow.is_enabled() and 'disabled' not in arrow.get_attribute('class'):
                            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", arrow)
                            time.sleep(0.5)
                            driver.execute_script("arguments[0].click();", arrow)
                            clicked_arrow = True
                            time.sleep(2)
                            break
                    
                    if clicked_arrow:
                        check_btns = driver.find_elements(By.XPATH, num_xpath)
                        if check_btns and check_btns[0].is_displayed():
                            driver.execute_script("arguments[0].click();", check_btns[0])
                            page_num += 1
                            time.sleep(1.5)
                            continue
                        else:
                            break 
                    else:
                        break 

        except Exception as e:
            print(f"🚨 [{category}] 수집 중 에러 발생: {e}")
            driver.get(url) 
            time.sleep(3)
            continue

finally:
    print("\n🎉 모든 크롤링 작업 완료. 자원을 정리합니다.")
    driver.quit()
    if cursor: cursor.close()
    if conn.is_connected(): conn.close()
    print("기아자동차 데이터 삽입 및 정리 완료")

데이터베이스에 연결 중... (DB: faq_data )
크롬 브라우저를 실행합니다...

[차량 구매] 카테고리 수집 준비 중...
✅ [차량 구매] 1페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 구매] 2페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 구매] 3페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 구매] 4페이지 수집 완료 (추출된 데이터: 8개)

[차량 정비] 카테고리 수집 준비 중...
✅ [차량 정비] 1페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 2페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 3페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 4페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 5페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 6페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 7페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 8페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 9페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 10페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 11페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 12페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 13페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 14페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 15페이지 수집 완료 (추출된 데이터: 10개)
✅ [차량 정비] 16페이지 수집 완료 (추출된 데이터: 4개)

[기아멤버스] 카테고리 수집 준비 중...
✅ [기아멤버스] 1페이지 수집 완료 (추출된 데이터: 4개)

[홈페이지] 카테고리 수집 준비 중...
✅ [홈페이지] 1페이지 수집 완료 (추출된 데이터: 1개)

[PBV] 카테고리 수집 준비 중...
✅ [PBV] 1페이지 수집 완료 (추출된 데이터: 10개)


In [ ]:
import os
import time
from dotenv import load_dotenv
import mysql.connector

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# 1. DB 설정 
# ---------------------------------------------------------
load_dotenv(override=True)
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'faq_data'), 
    'port': int(os.getenv('DB_PORT', 3306))
}

print("데이터베이스에 연결 중... (DB:", DB_CONFIG['database'], ")")
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

insert_query = '''
INSERT INTO hipass_faq (category_main, category_sub, question, answer) 
VALUES (%s, %s, %s, %s)
'''

# ---------------------------------------------------------
# 2. 브라우저 실행 및 접속
# ---------------------------------------------------------
print("크롬 브라우저를 실행합니다...")
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized') 

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

url = 'https://hipass.co.kr/board/selectFaqList.do'
driver.get(url)
time.sleep(3) 

try:
    print("\n[전체] 카테고리 수집을 시작합니다. (왕복 이동 및 이미지 맥락 유지 적용)")
    
    page_num = 1
    while True:
        # 1) 리스트 표가 나타날 때까지 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
        time.sleep(1) 
        
        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        row_count = len(rows)
        
        if row_count == 0 or (row_count == 1 and "없습니다" in rows[0].text):
            break 
            
        print(f"🔄 {page_num}페이지 수집 중... (총 {row_count}개 게시글)")

        # 2) 게시글 1개씩 클릭 -> 수집 -> 뒤로가기 반복
        for i in range(row_count):
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
                current_rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                row = current_rows[i]
                
                tds = row.find_elements(By.TAG_NAME, "td")
                if len(tds) < 4:
                    continue # 유효하지 않은 빈 줄 건너뛰기
                
                main_cat = tds[1].text.strip()
                sub_cat = tds[2].text.strip()
                
                subject_td = tds[3]
                question_text = subject_td.text.strip()
                
                # 상세 페이지 진입
                subject_link = subject_td.find_element(By.TAG_NAME, "a")
                driver.execute_script("arguments[0].click();", subject_link)
                
                # 3) 📸 상세 페이지 답변 및 이미지 링크 위치 매칭 추출
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.cm_board_view_contents")))
                time.sleep(0.5)
                
                soup = BeautifulSoup(driver.page_source, 'lxml')
                answer_box = soup.select_one("div.cm_board_view_contents")
                
                if answer_box:
                    # ✨ 텍스트를 뽑기 전에, img 태그를 찾아서 그 위치 그대로 텍스트 링크로 '치환(바꿔치기)' 합니다.
                    images = answer_box.find_all('img')
                    for img in images:
                        img_src = img.get('src')
                        if img_src:
                            if img_src.startswith('/'):
                                img_src = 'https://hipass.co.kr' + img_src
                            elif not img_src.startswith('http'):
                                img_src = 'https://hipass.co.kr/' + img_src
                            
                            # img 태그 자체를 [이미지: 주소] 형태의 텍스트로 덮어씌움
                            img.replace_with(f"\n\n[이미지: {img_src}]\n\n")
                    
                    # 치환이 모두 끝난 뒤에 텍스트를 추출하면 순서가 완벽히 유지됨!
                    answer_text = answer_box.get_text(separator='\n', strip=True)
                else:
                    answer_text = "내용 없음"
                
                # DB 저장
                cursor.execute(insert_query, (main_cat, sub_cat, question_text, answer_text))
                conn.commit()
                
                # 4) 뒤로 가기
                driver.back()
                time.sleep(1) 
                
                # 페이지 이탈 방지용 안전장치 (원래 페이지로 복귀)
                active_page_elem = driver.find_elements(By.CSS_SELECTOR, "a.page_link.active")
                if active_page_elem and active_page_elem[0].text.strip() != str(page_num):
                    driver.execute_script(f"movePage('{page_num}', '/board/selectFaqList.do')")
                    time.sleep(1.5)

            except Exception as e:
                print(f"   ⚠️ {i+1}번째 글 수집 중 오류 (건너뜀): {e}")
                driver.get(url)
                driver.execute_script(f"movePage('{page_num}', '/board/selectFaqList.do')")
                time.sleep(2)
                continue

        print(f"✅ {page_num}페이지 수집 완료")

        # ---------------------------------------------------------
        # 5. 페이징 (다음 숫자 추적)
        # ---------------------------------------------------------
        target_num_str = str(page_num + 1)
        page_links = driver.find_elements(By.XPATH, f"//a[contains(@class, 'page_link') and text()='{target_num_str}']")

        if page_links:
            driver.execute_script("arguments[0].click();", page_links[0])
            page_num += 1
            time.sleep(1.5)
        else:
            next_arrow = driver.find_elements(By.CSS_SELECTOR, "a.page_link.page_control.next")
            if next_arrow and next_arrow[0].is_displayed():
                driver.execute_script("arguments[0].click();", next_arrow[0])
                page_num += 1
                time.sleep(1.5)
            else:
                break 

except Exception as e:
    print(f"🚨 크롤링 중 치명적인 에러 발생: {e}")

finally:
    print("\n🎉 모든 크롤링 작업 완료. 자원을 정리합니다.")
    driver.quit()
    if cursor: cursor.close()
    if conn.is_connected(): conn.close()
    print("하이패스 데이터 삽입 및 정리 완료")

데이터베이스에 연결 중... (DB: faq_data )
크롬 브라우저를 실행합니다...


KeyboardInterrupt: 